# Qwen2.5 14B Colab Pro Server

이 노트북은 Google Colab Pro GPU에서 `Qwen/Qwen2.5-14B-Instruct`를 4bit 양자화로 실행하고, 로컬 VSCode/RAG 코드에서 호출할 수 있는 OpenAI 호환 API를 엽니다.

## 사용 흐름
1. Colab Pro에서 이 노트북을 엽고 위에서부터 셀을 순서대로 실행합니다.
2. ngrok URL이 출력되면 로컬 `.env`의 `LLM_API_BASE`에 `/v1`까지 포함해 넣습니다.
3. 로컬에서 `python cli.py`를 실행해 RAG 질의를 테스트합니다.

## Qwen2.5-14B-Instruct 사용 기준
- Colab Pro에서 L4/A100 GPU를 권장합니다. T4에서는 4bit도 느리거나 메모리가 부족할 수 있습니다.
- 로컬 `.env`의 `LLM_MODEL`도 `Qwen/Qwen2.5-14B-Instruct`로 맞춰야 로그와 요청 모델명이 일치합니다.
- ngrok URL이 새로 열리면 로컬 `.env`의 `LLM_API_BASE`만 새 주소로 교체합니다.


In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. 의존성 설치
# Qwen2.5 14B Instruct를 4bit로 로드합니다. transformers는 Qwen2.5 호환 버전을 사용합니다.
!pip uninstall -y transformers accelerate
!pip install "transformers>=4.46.0" "accelerate>=1.0.0" bitsandbytes fastapi uvicorn pyngrok nest_asyncio huggingface_hub requests

In [ ]:
Qwen2.5 14B 모델 다운로드에 Hugging Face 토큰이 필요할 수 있습니다. 공개 모델이지만 rate limit이나 계정 환경에 따라 토큰 입력을 권장합니다. 토큰은 노트북 파일에 저장되지 않습니다.import transformers
import torch

print("transformers", transformers.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

# 4. Hugging Face 토큰 설정(선택)
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Hugging Face token 입력(없으면 Enter): ")
if hf_token.strip():
    login(token=hf_token.strip())
    print("Hugging Face login OK")
else:
    print("Hugging Face token skipped")


In [ ]:
# 4. Hugging Face 로그인 (필요 시)
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Hugging Face token 입력(없으면 Enter): ")
if hf_token.strip():
    login(hf_token.strip())
    print("Hugging Face login complete")
else:
    print("skip Hugging Face login")

In [ ]:
# 5. 모델 로드
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

gc.collect()
torch.cuda.empty_cache()

# Colab Pro GPU에서도 안정성을 위해 기본은 4bit입니다.
# A100 40GB 이상에서 더 높은 품질/속도를 비교하려면 load_in_4bit=False 구조로 별도 테스트하세요.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=False,
)
model.eval()

print("loaded", MODEL_ID)
print("device", next(model.parameters()).device)

In [ ]:
# 6. 모델 단독 테스트
messages = [
    {"role": "system", "content": "너는 한국어로 간결하게 답한다."},
    {"role": "user", "content": "Qwen2.5 14B 서버 테스트입니다. 한 문장으로 응답하세요."},
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = output[0][inputs["input_ids"].shape[-1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

In [ ]:
# 7. OpenAI-compatible FastAPI 서버 실행
import json
import threading
import traceback
import uuid

import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field

nest_asyncio.apply()

app = FastAPI(title="Qwen2.5 14B Colab Pro OpenAI-compatible API")

class ChatRequest(BaseModel):
    model: str | None = None
    messages: list[dict]
    temperature: float = 0.1
    max_tokens: int = Field(default=768, alias="max_tokens")
    stream: bool = False

def build_prompt(messages: list[dict]) -> str:
    clean_messages = []
    for msg in messages:
        clean_messages.append({
            "role": msg.get("role", "user"),
            "content": str(msg.get("content", "")),
        })
    return tokenizer.apply_chat_template(
        clean_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def generate_text(req: ChatRequest) -> str:
    prompt = build_prompt(req.messages)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    max_new_tokens = max(1, min(int(req.max_tokens or 768), 1536))
    temperature = float(req.temperature or 0.0)
    do_sample = temperature > 0

    generation_kwargs = {
        **inputs,
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        generation_kwargs["temperature"] = max(temperature, 0.01)
        generation_kwargs["top_p"] = 0.9

    with torch.inference_mode():
        output = model.generate(**generation_kwargs)

    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

@app.get("/health")
def health():
    return {
        "ok": True,
        "model": MODEL_ID,
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }

@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model"}]}

@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    try:
        text = generate_text(req)
        completion_id = "chatcmpl-" + uuid.uuid4().hex

        if req.stream:
            def event_stream():
                chunk = {
                    "id": completion_id,
                    "object": "chat.completion.chunk",
                    "model": req.model or MODEL_ID,
                    "choices": [{"index": 0, "delta": {"content": text}, "finish_reason": None}],
                }
                yield "data: " + json.dumps(chunk, ensure_ascii=False) + "\\n\\n"
                yield "data: [DONE]\\n\\n"
            return StreamingResponse(event_stream(), media_type="text/event-stream")

        return {
            "id": completion_id,
            "object": "chat.completion",
            "model": req.model or MODEL_ID,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop",
            }],
        }
    except Exception as exc:
        traceback.print_exc()
        return JSONResponse(
            status_code=500,
            content={"error": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()},
        )

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("server started: http://localhost:8000")

In [ ]:
# 8. Colab 내부 API 테스트
import requests

payload = {
    "model": MODEL_ID,
    "messages": [{"role": "user", "content": "한국어로 한 문장만 답하세요. API 테스트입니다."}],
    "temperature": 0.1,
    "max_tokens": 128,
    "stream": False,
}

r = requests.post("http://localhost:8000/v1/chat/completions", json=payload, timeout=180)
print(r.status_code)
print(r.text[:2000])

## ngrok 공개 URL 열기

ngrok 토큰은 https://dashboard.ngrok.com/get-started/your-authtoken 에서 복사합니다. 계정 주인의 토큰을 입력해야 합니다.

In [ ]:
# 9. ngrok 터널 열기
from getpass import getpass
from pyngrok import ngrok

# 방법 A: 여기에 ngrok 토큰을 직접 넣기
# 주의: 노트북을 다른 사람 Drive에 저장하면 토큰도 파일에 남습니다.
NGROK_TOKEN = ""  # 예: "2abc..."

# 방법 B: 위 변수가 비어 있으면 실행 중 입력받기
if not NGROK_TOKEN.strip():
    NGROK_TOKEN = getpass("ngrok authtoken 입력: ")

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN.strip())
public_url = ngrok.connect(8000)
base_url = str(public_url).split(' -> ')[0].replace('NgrokTunnel: ', '').strip().strip('"')

print("PUBLIC_URL=", base_url)
print("LLM_API_BASE=", base_url + "/v1")

In [ ]:
# 10. 외부 URL 테스트
import requests

headers = {"ngrok-skip-browser-warning": "true"}

r = requests.get(base_url + "/health", headers=headers, timeout=30)
print("health", r.status_code, r.text)

r = requests.post(
    base_url + "/v1/chat/completions",
    headers=headers,
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": "한국어로 한 문장만 답하세요. 외부 연결 테스트입니다."}],
        "temperature": 0.1,
        "max_tokens": 128,
        "stream": False,
    },
    timeout=180,
)
print("chat", r.status_code)
print(r.text[:2000])

## 로컬 PC `.env` 설정

`LLM_API_BASE=`에 출력된 `/v1` 주소를 넣습니다.

```env
LLM_PROVIDER=remote_openai
LLM_MODEL=Qwen/Qwen2.5-14B-Instruct
LLM_API_BASE=https://xxxxx.ngrok-free.dev/v1
LLM_API_KEY=dummy
```

로컬 실행:

```cmd
conda activate p311_ragreport
cd /d "C:\\K-Safety Law RAG"
python cli.py
```